# 00 — Install & Setup

Install the `myllm` package and verify the public API surface.

> **Colab users:** Run the setup cell → wait for "Done!" → **Runtime → Restart runtime** → continue from cell 2.

In [ ]:
import os, sys, subprocess

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _install():
    IN_COLAB = _is_colab()
    if IN_COLAB:
        repo = 'MyLLM'
        if not os.path.exists(repo):
            print('Cloning repository...')
            r = subprocess.run(
                ['git', 'clone', 'https://github.com/silvaxxx1/MyLLM.git', repo],
                capture_output=True, text=True
            )
            if r.returncode != 0:
                print('Clone failed:\n', r.stderr); return
        print('Installing myllm...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', repo],
            capture_output=True, text=True
        )
    else:
        root = os.path.abspath(os.path.join(os.getcwd(), '..'))
        print(f'Installing from {root} ...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-e', root],
            capture_output=True, text=True
        )
    if r.returncode != 0:
        print('Install FAILED. Error output:')
        print(r.stdout[-3000:] if r.stdout else '')
        print(r.stderr[-3000:] if r.stderr else '')
    else:
        msg = 'Restart runtime now (Runtime → Restart runtime).' if IN_COLAB else 'Ready.'
        print('Done!', msg)

try:
    import myllm
    print(f'myllm {myllm.__version__} already installed.')
except ImportError:
    _install()

## 1. Verify the package loads

In [ ]:
import myllm
print('myllm version :', myllm.__version__)
print('public exports:', myllm.__all__)

## 2. Import styles — three ways to access the same API

In [ ]:
# Style 1 — flat top-level (most convenient)
from myllm import LLM, ModelConfig, GenerationConfig
from myllm import SFTTrainer, SFTTrainerConfig
from myllm import get_tokenizer

print('LLM             :', LLM)
print('ModelConfig     :', ModelConfig)
print('GenerationConfig:', GenerationConfig)
print('SFTTrainer      :', SFTTrainer)

In [ ]:
# Style 2 — submodule imports (HuggingFace style)
from myllm.train import SFTTrainer, DPOTrainer, PPOTrainer
from myllm.tokenizers import GPT2Tokenizer
from myllm.configs import ModelConfig, GenerationConfig

print('from myllm.train      :', SFTTrainer)
print('from myllm.tokenizers :', GPT2Tokenizer)
print('from myllm.configs    :', ModelConfig)

In [ ]:
# Style 3 — submodule attribute access
import myllm
print('myllm.train.SFTTrainer        :', myllm.train.SFTTrainer)
print('myllm.tokenizers.GPT2Tokenizer:', myllm.tokenizers.GPT2Tokenizer)

## 3. CLI

In [ ]:
!python -m myllm version
!python -m myllm models

## 4. Available model configs with memory estimates

In [ ]:
import torch
from myllm import ModelConfig

print(f'{"Model":<16}  {"Layers":>6}  {"Embd":>5}  {"Params":>8}  {"FP32 (GB)":>9}  {"FP16 (GB)":>9}')
print('-' * 65)
for name in ModelConfig.available_configs():
    cfg  = ModelConfig.from_name(name)
    fp32 = cfg.estimate_memory(dtype=torch.float32)
    fp16 = cfg.estimate_memory(dtype=torch.float16)
    print(f'{name:<16}  {cfg.n_layer:>6}  {cfg.n_embd:>5}  '
          f'{fp32["n_parameters"]/1e6:>7.1f}M  '
          f'{fp32["parameters_gb"]:>9.2f}  {fp16["parameters_gb"]:>9.2f}')